# Practical Session 01
## Part I - Instructor-led mini-lab
### Example 1 - Why not just use Python lists?
Take a look at the code below before running it.

Questions:

1. Will the two expressions produce the same result?
2. Which object represents a mathematical vector more naturally?
3. What does multiplication mean in each case?/

In [ ]:
import sys
import time
import numpy as np

In [ ]:
python_values = [1, 2, 3]
numpy_values = np.array([1, 2, 3])

print(2 * python_values)
print(2 * numpy_values)

Next, compare approximate storage requirements:

In [ ]:
n = 30_000_000

values_list = list(range(n))
values_array = np.arange(n, dtype=np.int64)

estimated_list_size = (
    sys.getsizeof(values_list)
    + n * sys.getsizeof(values_list[0])
)

print(f"Estimated list size: {estimated_list_size / 1024**2:.1f} MB")
print(f"NumPy array size:    {values_array.nbytes / 1024**2:.1f} MB")

The list estimate is approximate but useful. It shows the main issue: a list of Python integers is not stored as raw integer data. It is a container of references to Python objects. For large scientific arrays, this difference quickly becomes important.

Now compare a simple numerical transformation:

$$
y_i = 3x_i + 1.
$$

For the Python list, we need to loop over elements explicitly. For the NumPy array, we apply the expression to the whole array

In [ ]:
n = 3_000_000

values_list = list(range(n))
values_array = np.arange(n, dtype=np.int64)

def transform_list(values):
    return [3 * x + 1 for x in values]

def transform_array(values):
    return 3 * values + 1

def best_average_time(function, repeats=3, number=5):
    times = []
    for _ in range(repeats):
        start = time.perf_counter()
        for _ in range(number):
            function()
        stop = time.perf_counter()
        times.append((stop - start) / number)
    return min(times)

list_time = best_average_time(lambda: transform_list(values_list))
array_time = best_average_time(lambda: transform_array(values_array))

print(f"List comprehension: {1e3 * list_time:8.3f} ms")
print(f"NumPy expression:   {1e3 * array_time:8.3f} ms")
print(f"Speed-up factor:    {list_time / array_time:8.1f} x")

## Example 2 - Reading an array through its axes
A small collection of detector hits is stored as:

```text
hits[event, hit, component]
```

Before running each expression below, predict the shape of the result.

In [ ]:
import numpy as np

rng = np.random.default_rng(7)

# axes: event, hit, spatial component
hits = rng.normal(size=(4, 6, 3))

print("hits.shape:", hits.shape)
print("hits[0].shape:          ", hits[0].shape)
print("hits[:, 0].shape:       ", hits[:, 0].shape)
print("hits[..., 2].shape:     ", hits[..., 2].shape)
print("hits.mean(axis=1).shape:", hits.mean(axis=1).shape)
print("hits.mean(axis=(0, 1)).shape:", hits.mean(axis=(0, 1)).shape)

Questions:

1. Which axis represents independent events?
2. Which axis represents hits inside one event?
3. Which axis represents the spatial component?
4. Which axis should disappear if we compute one centre per event?

In [ ]:
# One centre per event.
event_centres = hits.mean(axis=1)

print("event_centres.shape:", event_centres.shape)

assert event_centres.shape == (4, 3)

## Example 3 - Broadcasting a calibration model

Suppose that every hit slot has its own gain and every coordinate has its own offset.

We want to compute:

$$
h^{\mathrm{corrected}}_{eic}=\left(h_{eic}-o_c\right)g_i.
$$

Here:  
$e$ - event  
$i$ - hit  
$c$ - component

In [ ]:
sensor_gain = np.linspace(0.9, 1.1, 6)               # one value per hit
coordinate_offset = np.array([0.2, -0.1, 0.05])      # x, y, z

print("hits.shape:             ", hits.shape)
print("sensor_gain.shape:      ", sensor_gain.shape)
print("coordinate_offset.shape:", coordinate_offset.shape)

We need broadcast-ready shapes:

```text
hits               : (event, hit, component)
coordinate_offset  : (    1,   1, component)
sensor_gain        : (    1, hit,         1)
```

In [ ]:
corrected_hits = (
    (hits - coordinate_offset[None, None, :])
    * sensor_gain[None, :, None]
)

print("corrected_hits.shape:", corrected_hits.shape)

assert corrected_hits.shape == hits.shape

Now add one event-dependent shift only to the $z$ component:

$$
h^{\mathrm{shifted}}_{ei2}=h^{\mathrm{corrected}}_{ei2}-d_e.
$$

In [ ]:
event_shift = np.array([0.0, 0.2, -0.1, 0.05])

shifted_hits = corrected_hits.copy()
shifted_hits[..., 2] -= event_shift[:, None]

print("event_shift[:, None].shape:", event_shift[:, None].shape)
print("shifted_hits.shape:        ", shifted_hits.shape)

assert shifted_hits.shape == corrected_hits.shape

## Example 4 - A batched quadratic form

For each particle, let:

$$
p = (E, p_x, p_y, p_z).
$$

With the Minkowski metric

$$
\eta = \mathrm{diag}(1, -1, -1, -1),
$$

the invariant mass satisfies:

$$
m_b^2 = p_{bi}\eta_{ij}p_{bj}.
$$

The index $b$ labels particles and should remain. The indices $i,j$ are contracted.

In [ ]:
spatial_momentum = rng.normal(size=(5, 3))
particle_mass = rng.uniform(0.2, 2.0, size=5)

energy = np.sqrt(
    particle_mass**2
    + np.sum(spatial_momentum**2, axis=1)
)

four_momentum = np.column_stack((energy, spatial_momentum))
metric = np.diag([1.0, -1.0, -1.0, -1.0])

print("four_momentum.shape:", four_momentum.shape)
print("metric.shape:       ", metric.shape)

In [ ]:
mass_squared = np.einsum(
    "bi,ij,bj->b",
    four_momentum,
    metric,
    four_momentum,
)

reconstructed_mass = np.sqrt(mass_squared)

print("original masses:      ", particle_mass)
print("reconstructed masses: ", reconstructed_mass)

assert mass_squared.shape == (5,)
assert np.allclose(reconstructed_mass, particle_mass)

The same result can also be obtained in two steps: first apply the metric, then reduce over the component axis.

In [ ]:
metric_times_p = four_momentum @ metric
mass_squared_alt = np.sum(metric_times_p * four_momentum, axis=1)

print(mass_squared_alt)

assert np.allclose(mass_squared_alt, mass_squared)

Questions:

1. Which index labels independent particles?
2. Which indices disappear?
3. Why is the result one-dimensional?

## Example 5 - The view that changes the original data

Some NumPy operations create views of the same data. This is efficient, but it can also lead to accidental modification of the original array.

In [ ]:
raw_data = np.arange(24.0).reshape(4, 6)
window = raw_data[:, 2:5]

print("raw_data before:")
print(raw_data)

print("\nwindow:")
print(window)

print("\nShare memory?", np.shares_memory(raw_data, window))

In [ ]:
window[0, 0] = -999.0

print("raw_data after modifying window:")
print(raw_data)

The slice `raw_data[:, 2:5]` is a view. It does not own independent data.

To work safely on an independent array, use `.copy()`.

In [ ]:
raw_data = np.arange(24.0).reshape(4, 6)
safe_window = raw_data[:, 2:5].copy()

safe_window[0, 0] = -999.0

print("safe_window:")
print(safe_window)

print("\nraw_data:")
print(raw_data)

print("\nShare memory?", np.shares_memory(raw_data, safe_window))

## Part II - Independent work

### Task 1 - A Calibration Pipeline with Three Independent Axes

A detector records a short signal from several channels during each event. The array

```text
raw[event, channel, sample]
```
has shape `(12, 7, 64)`.

Each channel has its own pedestal $p_c$ and gain $g_c$. In addition, every event has a common drift $d_e$. The calibrated signal is

$$
s_{ecs} = (r_{ecs}-p_c-d_e)g_c
$$
Use the following data:

In [ ]:
import numpy as np

rng = np.random.default_rng(2026)

n_events = 12
n_channels = 7
n_samples = 64

raw = rng.normal(
    loc=1000.0,
    scale=20.0,
    size=(n_events, n_channels, n_samples),
)

pedestal = rng.normal(
    loc=995.0,
    scale=2.0,
    size=n_channels,
)

gain = rng.uniform(
    0.8,
    1.2,
    size=n_channels,
)

event_drift = rng.normal(
    0.0,
    1.0,
    size=n_events,
)

Your tasks are:

1. Write one vectorized expression that calculates the calibrated signal. Do not use loops or explicitly repeat any calibration array.
2. Add comments showing the broadcast-ready shape of `pedestal`, `gain`, and `event_drift`.
3. Verify that the calibrated signal has the same shape as `raw`.
4. Calculate the temporal mean of every event-channel trace. The result should have shape (`n_events`, `n_channels`).
5. Subtract this temporal mean from each corresponding trace while preserving the original three-dimensional shape.
6. Verify numerically that the temporal means of the centred traces are approximately zero.
7. Find the event index, channel index, and sample index at which the centred signal has the largest absolute value.

**Optional extension**: Package the calibration operation in a function that checks all input shapes and raises an informative error when they are inconsistent.

In [ ]:
#####################
# Your solution here
#####################

# The input array has shape:
# raw[event, channel, sample] -> (n_events, n_channels, n_samples)

# ------------------------------------------------------------
# Point 1 and 2
# Vectorized calibration using broadcasting
# ------------------------------------------------------------

# Broadcast-ready shapes:
# raw:                        (n_events, n_channels, n_samples)
# pedestal[None, :, None]:    (1,        n_channels, 1)
# gain[None, :, None]:        (1,        n_channels, 1)
# event_drift[:, None, None]: (n_events, 1,          1)

calibrated = (
    raw
    - pedestal[None, :, None]
    - event_drift[:, None, None]
) * gain[None, :, None]


# ------------------------------------------------------------
# Point 3
# Check that the calibrated signal has the same shape as raw
# ------------------------------------------------------------

print("raw shape:        ", raw.shape)
print("calibrated shape: ", calibrated.shape)

assert calibrated.shape == raw.shape


# ------------------------------------------------------------
# Point 4
# Calculate the temporal mean of each event-channel trace
# ------------------------------------------------------------

# We average over the sample axis, which is axis 2.
# Result shape: (n_events, n_channels)

trace_mean = calibrated.mean(axis=2)

print("trace mean shape: ", trace_mean.shape)

assert trace_mean.shape == (n_events, n_channels)


# ------------------------------------------------------------
# Point 5
# Subtract the temporal mean from each corresponding trace
# ------------------------------------------------------------

# trace_mean has shape (n_events, n_channels)
# trace_mean[:, :, None] has shape (n_events, n_channels, 1)
# This allows broadcasting back to the full 3D shape.

centered = calibrated - trace_mean[:, :, None]

print("centered shape:   ", centered.shape)

assert centered.shape == raw.shape


# ------------------------------------------------------------
# Point 6
# Verify that temporal means of centered traces are close to zero
# ------------------------------------------------------------

centered_means = centered.mean(axis=2)

print("centered means shape: ", centered_means.shape)
print("Means are close to zero:", np.allclose(centered_means, 0.0))
print("Maximum absolute mean after centering:",
      np.max(np.abs(centered_means)))

assert np.allclose(centered_means, 0.0)


# ------------------------------------------------------------
# Point 7
# Find the position of the largest absolute value
# ------------------------------------------------------------

flat_index = np.argmax(np.abs(centered))

event_idx, channel_idx, sample_idx = np.unravel_index(
    flat_index,
    centered.shape
)

print("Largest absolute value location:")
print("event index:   ", event_idx)
print("channel index: ", channel_idx)
print("sample index:  ", sample_idx)

print("centered value:",
      centered[event_idx, channel_idx, sample_idx])
print("absolute value:",
      np.abs(centered[event_idx, channel_idx, sample_idx]))


# ------------------------------------------------------------
# Optional extension
# Package the calibration step into a simple function
# ------------------------------------------------------------

def calibrate_signal(raw, pedestal, gain, event_drift):
    # Check the main signal shape.
    if raw.ndim != 3:
        raise ValueError(
            f"raw must have shape (n_events, n_channels, n_samples), "
            f"but got {raw.shape}"
        )

    n_events, n_channels, n_samples = raw.shape

    # Check calibration array shapes.
    if pedestal.shape != (n_channels,):
        raise ValueError(
            f"pedestal must have shape ({n_channels},), "
            f"but got {pedestal.shape}"
        )

    if gain.shape != (n_channels,):
        raise ValueError(
            f"gain must have shape ({n_channels},), "
            f"but got {gain.shape}"
        )

    if event_drift.shape != (n_events,):
        raise ValueError(
            f"event_drift must have shape ({n_events},), "
            f"but got {event_drift.shape}"
        )

    # Vectorized calibration.
    return (
        raw
        - pedestal[None, :, None]
        - event_drift[:, None, None]
    ) * gain[None, :, None]


# Example use of the optional function.
calibrated_from_function = calibrate_signal(
    raw,
    pedestal,
    gain,
    event_drift
)

assert calibrated_from_function.shape == raw.shape
assert np.allclose(calibrated_from_function, calibrated)

print("Optional function check passed.")

## Task 2 - The Case of the Corrupted Raw Data

A preprocessing step extracts samples 3–8 from every detector trace and centres them:

In [ ]:
raw = np.arange(6 * 12, dtype=float).reshape(6, 12)
raw_before_processing = raw.copy()

window = raw[:, 3:9]
window -= window.mean(axis=1, keepdims=True)

After this operation, a colleague notices that raw no longer contains the original measurements.

Your tasks are:

1. Compare `raw` with `raw_before_processing` and identify which entries changed.
2. Explain why modifying `window` also modified `raw`.
3. Repair the preprocessing step so that the extracted window can be modified safely while `raw` remains unchanged.
4. Use `np.shares_memory` to investigate whether each of the following is normally a view or a copy:

In [ ]:
raw[:, 2:8]
raw[:, ::2]
raw.T
raw.reshape(-1)
raw.T.reshape(-1)
raw[:, [2, 4, 6]]
raw[raw > 20]
raw + 0.0

5. For every expression, record its shape and whether it shares memory with `raw`.
6. Create an independent centred window, flatten it into a one-dimensional solver state, and then reconstruct its original two-dimensional shape.
7. Verify that reconstruction preserves all values exactly.

Conclude with two or three sentences explaining why an operation that is efficient because it avoids copying can also create a correctness problem.

In [ ]:
#####################
# Your solution here
#####################

# ------------------------------------------------------------
# Point 1
# Compare raw with raw_before_processing and identify changed entries
# ------------------------------------------------------------

changed_mask = raw != raw_before_processing

changed_rows, changed_cols = np.where(changed_mask)

changed_entries = np.column_stack(
    (
        changed_rows,
        changed_cols,
        raw_before_processing[changed_mask],
        raw[changed_mask],
    )
)

print("Changed entries as [row, column, before, after]:")
print(changed_entries)

print("Number of changed entries:", changed_entries.shape[0])
print("Changed columns:", np.unique(changed_cols))

assert np.all(changed_mask[:, 3:9])
assert not np.any(changed_mask[:, :3])
assert not np.any(changed_mask[:, 9:])


# ------------------------------------------------------------
# Point 2
# Explain why modifying window also modified raw
# ------------------------------------------------------------

print()
print("Why did raw change?")
print("window = raw[:, 3:9] is a slicing operation.")
print("This normally creates a view, not an independent copy.")
print("Therefore, window and raw share the same underlying memory.")

print("Does window share memory with raw?",
      np.shares_memory(window, raw))

assert np.shares_memory(window, raw)


# ------------------------------------------------------------
# Point 3
# Repair the preprocessing step so that raw remains unchanged
# ------------------------------------------------------------

# Restore raw from the saved original data.
raw = raw_before_processing.copy()
raw_check = raw.copy()

# The extracted window is copied before in-place modification.
safe_window = raw[:, 3:9].copy()
safe_window -= safe_window.mean(axis=1, keepdims=True)

print()
print("Safe preprocessing:")
print("raw remained unchanged:",
      np.array_equal(raw, raw_check))
print("safe_window shares memory with raw:",
      np.shares_memory(safe_window, raw))

assert np.array_equal(raw, raw_check)
assert not np.shares_memory(safe_window, raw)


# ------------------------------------------------------------
# Point 4 and 5
# Investigate views and copies using np.shares_memory
# ------------------------------------------------------------

memory_tests = [
    ("raw[:, 2:8]", raw[:, 2:8]),
    ("raw[:, ::2]", raw[:, ::2]),
    ("raw.T", raw.T),
    ("raw.reshape(-1)", raw.reshape(-1)),
    ("raw.T.reshape(-1)", raw.T.reshape(-1)),
    ("raw[:, [2, 4, 6]]", raw[:, [2, 4, 6]]),
    ("raw[raw > 20]", raw[raw > 20]),
    ("raw + 0.0", raw + 0.0),
]

print()
print("Memory sharing investigation:")
print(f"{'Expression':22s} {'Shape':12s} {'Shares memory with raw'}")

for expression, result in memory_tests:
    shares = np.shares_memory(result, raw)
    print(f"{expression:22s} {str(result.shape):12s} {shares}")


# ------------------------------------------------------------
# Point 6
# Create an independent centred window, flatten it, and reconstruct it
# ------------------------------------------------------------

# Start from an independent copy of the selected window.
independent_window = raw[:, 3:9].copy()

# Centre each row independently.
centred_window = (
    independent_window
    - independent_window.mean(axis=1, keepdims=True)
)

# Flatten into a one-dimensional solver state.
solver_state = centred_window.reshape(-1)

# Reconstruct the original two-dimensional shape.
reconstructed_window = solver_state.reshape(centred_window.shape)

print()
print("Centred window shape:", centred_window.shape)
print("Solver state shape:", solver_state.shape)
print("Reconstructed window shape:", reconstructed_window.shape)

assert centred_window.shape == (6, 6)
assert solver_state.shape == (36,)
assert reconstructed_window.shape == centred_window.shape


# ------------------------------------------------------------
# Point 7
# Verify that reconstruction preserves all values exactly
# ------------------------------------------------------------

print("Reconstruction preserves all values exactly:",
      np.array_equal(reconstructed_window, centred_window))

assert np.array_equal(reconstructed_window, centred_window)

Conclusion

Views are efficient because they avoid copying data and only reinterpret existing memory. However, this also means that in-place modification of a view can silently modify the original array. When the original data must be preserved, an explicit copy is safer than relying on a view.

## Task 3 - A Resonance Hidden in a Combinatorial Background

A simplified detector event contains two collections of approximately massless particle candidates with opposite electric charges. Each candidate is represented by a four-momentum

$$
p = (E, p_x, p_y, p_z)
$$

One positive-negative pair was generated from a particle with mass

$$
M_{\text{target}} = 91.1876
$$

The remaining combinations form a combinatorial background.

Use the following data-generation cell:

In [ ]:
import numpy as np

rng = np.random.default_rng(31415)
target_mass = 91.1876

def random_massless_four_momenta(n):
    directions = rng.normal(size=(n, 3))
    directions /= np.linalg.norm(
        directions,
        axis=1,
        keepdims=True,
    )

    energies = rng.uniform(10.0, 80.0, size=n)

    return np.column_stack(
        (energies, energies[:, None] * directions)
    )

positive = random_massless_four_momenta(28)
negative = random_massless_four_momenta(31)

direction = np.array([0.3, -0.4, 0.866025403784])
direction /= np.linalg.norm(direction)

energy = target_mass / 2.0

positive_signal = np.r_[energy, energy * direction]
negative_signal = np.r_[energy, -energy * direction]

positive = np.vstack((positive, positive_signal))
negative = np.vstack((negative, negative_signal))

positive = positive[rng.permutation(len(positive))]
negative = negative[rng.permutation(len(negative))]

metric = np.diag([1.0, -1.0, -1.0, -1.0])

For a positive candidate $i$ and a negative candidate $j$, define

$$
P_{ij} = p_i^{(+)} + p_j^{(-)}
$$

and

$$
m_{ij}^{2}=P_{ij}^{\mu}\,\eta_{\mu\nu}\,P_{ij}^{\nu},\qquad
\eta = \operatorname{diag}(1,-1,-1,-1).
$$

Your tasks are:

1. State the meaning of both axes of `positive` and `negative`.
2. Use broadcasting to construct the four-momentum of every possible positive-negative pair. The result should have shape
```text
(number of positive candidates,
 number of negative candidates,
 4)
```
3. Use `np.einsum` to calculate the complete matrix of invariant masses. Do not loop over candidate pairs.
4. Protect the square root from very small negative values caused by floating-point round-off.
5. Find the pair whose invariant mass is closest to `target_mass`.
6. Report:
    - the two candidate indices;
    - their reconstructed invariant mass;
    - the absolute mass difference from the target;
    - the magnitude of the total three-momentum of the pair.
7. Find all candidate pairs within 0.5 units of the target mass.
8. For every positive candidate, find the negative candidate that gives the closest mass to the target. Be careful to reduce over the correct axis.
9. Validate the invariant mass of the best pair using the direct scalar expression

$$
m^2 = E^2 - p_x^2 - p_y^2 - p_z^2
$$

No nested Python loops may be used.

**Optional extension**: Repeat the calculation after exchanging the positive and negative collections. Explain why the new mass matrix should be the transpose of the original one.

Useful NumPy tools may include `np.argmin`, `np.unravel_index`, `np.where`, and `np.clip`.

In [ ]:
#####################
# Your solution here
#####################

# ------------------------------------------------------------
# Point 1
# State the meaning of both axes of positive and negative
# ------------------------------------------------------------

# positive has shape (number of positive candidates, 4)
# Axis 0: positive particle candidate index
# Axis 1: four-momentum component (E, px, py, pz)

# negative has shape (number of negative candidates, 4)
# Axis 0: negative particle candidate index
# Axis 1: four-momentum component (E, px, py, pz)

print("positive shape:", positive.shape)
print("negative shape:", negative.shape)

assert positive.ndim == 2
assert negative.ndim == 2
assert positive.shape[1] == 4
assert negative.shape[1] == 4


# ------------------------------------------------------------
# Point 2
# Construct the four-momentum of every positive-negative pair
# ------------------------------------------------------------

# Broadcast-ready shapes:
# positive[:, None, :] -> (number of positive candidates, 1, 4)
# negative[None, :, :] -> (1, number of negative candidates, 4)
#
# Result:
# pair_four_momentum[i, j, :] = positive[i, :] + negative[j, :]

pair_four_momentum = positive[:, None, :] + negative[None, :, :]

print("pair_four_momentum shape:", pair_four_momentum.shape)

assert pair_four_momentum.shape == (
    positive.shape[0],
    negative.shape[0],
    4,
)


# ------------------------------------------------------------
# Point 3
# Calculate the full invariant mass matrix using np.einsum
# ------------------------------------------------------------

# pair_four_momentum has shape (n_positive, n_negative, 4)
# metric has shape (4, 4)
#
# invariant_mass_squared[i, j] =
#     P[i, j, mu] * metric[mu, nu] * P[i, j, nu]

invariant_mass_squared = np.einsum(
    "ijm,mn,ijn->ij",
    pair_four_momentum,
    metric,
    pair_four_momentum,
)

print("invariant_mass_squared shape:", invariant_mass_squared.shape)

assert invariant_mass_squared.shape == (
    positive.shape[0],
    negative.shape[0],
)


# ------------------------------------------------------------
# Point 4
# Protect the square root from tiny negative round-off errors
# ------------------------------------------------------------

# Very small negative values may appear because of floating-point arithmetic.
# They are clipped to zero before taking the square root.

safe_mass_squared = np.clip(
    invariant_mass_squared,
    a_min=0.0,
    a_max=None,
)

invariant_mass = np.sqrt(safe_mass_squared)

print("invariant_mass shape:", invariant_mass.shape)

assert invariant_mass.shape == invariant_mass_squared.shape
assert np.all(safe_mass_squared >= 0.0)


# ------------------------------------------------------------
# Point 5
# Find the pair whose invariant mass is closest to target_mass
# ------------------------------------------------------------

mass_difference = np.abs(invariant_mass - target_mass)

best_flat_index = np.argmin(mass_difference)

best_positive_index, best_negative_index = np.unravel_index(
    best_flat_index,
    mass_difference.shape,
)

best_mass = invariant_mass[
    best_positive_index,
    best_negative_index,
]

best_mass_difference = mass_difference[
    best_positive_index,
    best_negative_index,
]


# ------------------------------------------------------------
# Point 6
# Report the best pair and useful diagnostics
# ------------------------------------------------------------

best_pair_four_momentum = pair_four_momentum[
    best_positive_index,
    best_negative_index,
]

best_pair_three_momentum = best_pair_four_momentum[1:]

best_pair_three_momentum_magnitude = np.linalg.norm(
    best_pair_three_momentum
)

print()
print("Best pair:")
print("positive candidate index:", best_positive_index)
print("negative candidate index:", best_negative_index)
print("reconstructed invariant mass:", best_mass)
print("absolute mass difference from target:", best_mass_difference)
print("magnitude of total three-momentum:",
      best_pair_three_momentum_magnitude)


# ------------------------------------------------------------
# Point 7
# Find all candidate pairs within 0.5 units of target_mass
# ------------------------------------------------------------

close_positive_indices, close_negative_indices = np.where(
    mass_difference <= 0.5
)

close_pairs = np.column_stack(
    (
        close_positive_indices,
        close_negative_indices,
        invariant_mass[close_positive_indices, close_negative_indices],
        mass_difference[close_positive_indices, close_negative_indices],
    )
)

print()
print("Pairs within 0.5 units of the target mass:")
print("Columns: positive_index, negative_index, mass, mass_difference")
print(close_pairs)

print("Number of close pairs:", close_pairs.shape[0])


# ------------------------------------------------------------
# Point 8
# For every positive candidate, find the closest negative candidate
# ------------------------------------------------------------

# mass_difference has shape (n_positive, n_negative).
# For each positive candidate, we reduce over axis 1,
# because axis 1 corresponds to negative candidates.

best_negative_for_each_positive = np.argmin(
    mass_difference,
    axis=1,
)

positive_indices = np.arange(positive.shape[0])

best_mass_for_each_positive = invariant_mass[
    positive_indices,
    best_negative_for_each_positive,
]

best_difference_for_each_positive = mass_difference[
    positive_indices,
    best_negative_for_each_positive,
]

best_match_per_positive = np.column_stack(
    (
        positive_indices,
        best_negative_for_each_positive,
        best_mass_for_each_positive,
        best_difference_for_each_positive,
    )
)

print()
print("Best negative candidate for every positive candidate:")
print("Columns: positive_index, best_negative_index, mass, mass_difference")
print(best_match_per_positive)

assert best_negative_for_each_positive.shape == (positive.shape[0],)


# ------------------------------------------------------------
# Point 9
# Validate the best invariant mass using the direct scalar formula
# ------------------------------------------------------------

E = best_pair_four_momentum[0]
px = best_pair_four_momentum[1]
py = best_pair_four_momentum[2]
pz = best_pair_four_momentum[3]

direct_mass_squared = E**2 - px**2 - py**2 - pz**2
direct_mass = np.sqrt(np.clip(direct_mass_squared, 0.0, None))

print()
print("Direct validation for the best pair:")
print("mass from np.einsum:", best_mass)
print("mass from direct formula:", direct_mass)
print("values agree:", np.allclose(best_mass, direct_mass))

assert np.allclose(best_mass, direct_mass)


# ------------------------------------------------------------
# Optional extension
# Repeat the calculation after exchanging positive and negative collections
# ------------------------------------------------------------

# Now the first axis corresponds to negative candidates,
# and the second axis corresponds to positive candidates.
#
# Shape:
# pair_four_momentum_swapped[j, i, :] =
#     negative[j, :] + positive[i, :]

pair_four_momentum_swapped = negative[:, None, :] + positive[None, :, :]

print()
print("Swapped pair_four_momentum shape:",
      pair_four_momentum_swapped.shape)

assert pair_four_momentum_swapped.shape == (
    negative.shape[0],
    positive.shape[0],
    4,
)


# Calculate the invariant mass matrix again.

invariant_mass_squared_swapped = np.einsum(
    "ijm,mn,ijn->ij",
    pair_four_momentum_swapped,
    metric,
    pair_four_momentum_swapped,
)

safe_mass_squared_swapped = np.clip(
    invariant_mass_squared_swapped,
    a_min=0.0,
    a_max=None,
)

invariant_mass_swapped = np.sqrt(safe_mass_squared_swapped)

print("Swapped invariant_mass shape:",
      invariant_mass_swapped.shape)

assert invariant_mass_swapped.shape == (
    negative.shape[0],
    positive.shape[0],
)


# The swapped mass matrix should be the transpose of the original one.

print("Swapped mass matrix equals original transpose:",
      np.allclose(invariant_mass_swapped, invariant_mass.T))

assert np.allclose(invariant_mass_swapped, invariant_mass.T)


# Explanation:
#
# The original matrix has entries:
# invariant_mass[i, j] = mass(positive[i] + negative[j])
#
# After swapping the collections, the new matrix has entries:
# invariant_mass_swapped[j, i] = mass(negative[j] + positive[i])
#
# Four-momentum addition is commutative, so:
# positive[i] + negative[j] = negative[j] + positive[i]
#
# Therefore, the numerical mass values are the same, but the two candidate
# axes are exchanged. This is why the swapped matrix is the transpose of
# the original mass matrix.